# 05 - MongoDB Search Lab

Goal: run vector and keyword search separately in MongoDB Atlas.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from pymongo import MongoClient
import voyageai

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(project_root / "django_app" / ".env")

MONGODB_URI = os.getenv("MONGODB_URI", "").strip()
api_key = os.getenv("VOYAGE_API_KEY", "").strip()
assert MONGODB_URI and api_key, "Set MONGODB_URI and VOYAGE_API_KEY first"

client = MongoClient(MONGODB_URI)
db = client[os.getenv("DB_NAME", "pythonasia2026_workshop")]
collection = db[os.getenv("COLLECTION_NAME", "recommendation_movies")]
vo = voyageai.Client(api_key=api_key)

In [ ]:
query = "space exploration mission"
query_vector = vo.embed([query], model="voyage-4-large").embeddings[0]

vector_pipeline = [
    {
        "$vectorSearch": {
            "index": "vector_index",
            "path": "embedding",
            "queryVector": query_vector,
            "numCandidates": 100,
            "limit": 5,
        }
    },
    {"$project": {"_id": 0, "title": 1, "score": {"$meta": "vectorSearchScore"}}},
]

keyword_pipeline = [
    {
        "$search": {
            "index": "default",
            "text": {"query": query, "path": ["title", "plot"]},
        }
    },
    {"$limit": 5},
    {"$project": {"_id": 0, "title": 1, "score": {"$meta": "searchScore"}}},
]

vector_results = list(collection.aggregate(vector_pipeline))
keyword_results = list(collection.aggregate(keyword_pipeline))

print("Vector results")
for item in vector_results:
    print(item)

print("\nKeyword results")
for item in keyword_results:
    print(item)